# Weak labels for 12 knee findings, and the threshold wall

[Version 1 of this notebook](https://www.kaggle.com/code/nekkon/weak-labels-for-all-12-knee-mri-findings)
extracted all twelve findings from all 4,407 reports with a multilingual keyword matcher and
scored it against the 58 labelled studies. Mean balanced accuracy 0.672, with medial meniscus
at 0.558 -- barely better than a coin flip -- and I wrote at the time that the meniscus findings
were close to unrecoverable from text.

That was wrong, and this version shows why. Replacing the keyword matcher with a **cross-lingual
entailment model** takes medial meniscus from 0.558 to **0.87 AUC**, and the model demonstrably
binds the *side*: its medial score is at chance on the lateral label. The keyword matcher never
did -- its medial score predicts the lateral label *better* than its own.

Then the notebook runs into the thing that actually limits this competition. The better scores
rank better (mean AUC 0.672 -> 0.75), but turning them into binary labels at a threshold fitted
on 58 studies gives back **the entire gain**. That is not a modelling failure, it is an
information limit, and it changes what you should do with the output.

**What you get at the end:** `weak_labels_v2.csv` -- 4,407 rows, per-finding scores from both
methods plus a blend, as probabilities rather than 0/1, with a per-finding reliability column.

In [ ]:
import glob, os, re, pickle, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

warnings.filterwarnings('ignore')

INK, MUTED, GRID = '#1a1a1a', '#5c5c5c', '#dcdcdc'
KWC, NLIC, BLEND, WARN = '#2a78d6', '#eb6834', '#1baf7a', '#e34948'
plt.rcParams.update({'figure.facecolor': 'white', 'axes.facecolor': 'white',
                     'axes.spines.top': False, 'axes.spines.right': False,
                     'axes.edgecolor': GRID, 'font.size': 11, 'text.color': INK,
                     'axes.labelcolor': MUTED, 'xtick.color': MUTED, 'ytick.color': MUTED,
                     'axes.titlesize': 13, 'axes.titleweight': 'bold',
                     'axes.titlelocation': 'left', 'axes.titlepad': 12})

P = glob.glob('/kaggle/input/**/train.csv', recursive=True)[0].rsplit('/', 1)[0]
train = pd.read_csv(f'{P}/train.csv')
LABELS = [c for c in train.columns if c not in ('StudyInstanceUID', 'Report')]
gold = train[train[LABELS].notna().all(axis=1)].reset_index(drop=True)

print(f'{len(train):,} studies, {len(gold)} of them labelled, {len(LABELS)} findings')
print('device:', 'cuda' if torch.cuda.is_available() else 'cpu')

## 1. The v1 baseline, unchanged

This is the keyword extractor from version 1, copied verbatim so the comparison below is against
exactly the floor that notebook reported. It carries a vocabulary per finding per language, and
negation windows that run left for the Latin-script languages and right for Turkish
(`efuzyon izlenmedi` -- "effusion was not observed" -- negates *after* the term).

Twelve findings times seven languages is the maintenance surface, and it is the reason to want
something that does not need a vocabulary at all.

In [ ]:
NEG_L = (r"\b(?:no|not|without|absent|negative for|geen|zonder|kein[e]?|ohne|sin|sans|"
         r"senza|non|nessun[ao]?|δεν|χωρ[ιί]ς)\b")
NEG_R = r"\b(?:izlenmedi|saptanmad[iı]|yok|g[oö]r[uü]lmedi|mevcut de[gğ]il|izlenmemi[sş]tir)\b"

TEAR = (r"tear|torn|rupture|ruptur|rotura|ruptura|scheur|ruptuur|riss|d[ée]chirure|"
        r"lesione|y[iı]rt[iı]k|rupt[uü]r|ρ[ήη]ξη")
MENISC = r"menisc|menisk|μηνίσκ"
MEDIAL = r"medial|interno|internal|mediale|binnen|innen|i[cç] |[έε]σω"
LATERAL = r"lateral|externo|external|laterale|buiten|au[sß]en|[έε]ξω"

TERMS = {
    "ACL": (r"anterior cruciate|\bacl\b|ligamento cruzado anterior|\blca\b|"
            r"voorste kruisband|\bvkb\b|vorderes kreuzband|ligament crois[ée] ant[ée]rieur|"
            r"[oö]n [cç]apraz|πρ[όο]σθιο[υς]? χιαστο", TEAR),
    "MCL": (r"medial collateral|\bmcl\b|ligamento colateral (?:medial|interno)|"
            r"mediale collaterale|innenband|mediales? kollateral|"
            r"i[cç] yan ba[gğ]|[έε]σω πλ[άα]γιο", TEAR),
    "Medial Meniscus": (rf"(?:{MEDIAL})\W+(?:\w+\W+){{0,2}}(?:{MENISC})|"
                        rf"(?:{MENISC})\w*\W+(?:\w+\W+){{0,2}}(?:{MEDIAL})", TEAR),
    "Lateral Meniscus": (rf"(?:{LATERAL})\W+(?:\w+\W+){{0,2}}(?:{MENISC})|"
                         rf"(?:{MENISC})\w*\W+(?:\w+\W+){{0,2}}(?:{LATERAL})", TEAR),
    "Medial OA": (rf"(?:osteoarthr|arthros|artros|gonarthros|artrosis|chondropath|"
                  rf"condropat|kraakbeenlijden|kn?orpelschaden|osteoartrit|"
                  rf"kondropati|αρθρ[ίι]τιδα)", MEDIAL),
    "Lateral OA": (rf"(?:osteoarthr|arthros|artros|gonarthros|artrosis|chondropath|"
                   rf"condropat|kraakbeenlijden|kn?orpelschaden|osteoartrit|"
                   rf"kondropati|αρθρ[ίι]τιδα)", LATERAL),
    "PF OA": (r"osteoarthr|arthros|artros|artrosis|chondropath|condropat|chondral|"
              r"condral|kraakbeenlijden|kn?orpelschaden|kondromalaz|chondromalac|"
              r"condromalac|osteoartrit|αρθρ[ίι]τιδα|χονδροπ[άα]θ",
              r"patellofemoral|femoropatell|patello-femoral|retropatellar|"
              r"femoropatelar|patell|patella|rotul|trochlea|tr[oó]clea|επιγονατιδ"),
    "Effusion": (r"effusion|joint fluid|derrame|gewrichtsvocht|erguss|[ée]panchement|"
                 r"versamento|ef[uü]zyon|eklem s[iı]v[iı]|υγρ[οό]", None),
    "Synovitis": (r"synovit|sinovit|synovial (?:thicken|proliferat|hypertroph)|"
                  r"synoviale? (?:verdikking|proliferat)|synovialitis|υμεν[ίι]τιδα|"
                  r"plica synovial", None),
    "Baker's": (r"baker|popliteal cyst|quiste popl[ií]teo|bakercyste|poplitea?le? cyste|"
                r"bakerzyste|kyste popliet|baker kist|popliteal kist|κ[ύυ]στη baker", None),
    "Contusion": (r"contusion|contusi[oó]n|bone (?:marrow )?(?:o)?edema|"
                  r"botoedeem|beenmergoedeem|knochenmark[soö]dem|edema (?:de m[ée]dula|[oó]seo)|"
                  r"kemik ili[gğ]i [oö]dem|[οο]?[ίι]δημα μυελο|bone bruise", None),
    "Fracture": (r"fracture|fractura|fractuur|fraktur|frattura|k[iı]r[iı]k|"
                 r"avulsion|avulsi[oó]n|κ[άα]ταγμα", None),
}


def hit(text, pattern, need=None, window=110):
    t = " " + re.sub(r"\s+", " ", text.lower()) + " "
    for m in re.finditer(pattern, t):
        left = t[max(0, m.start() - window):m.start()]
        right = t[m.end():m.end() + window]
        if re.search(NEG_L + r"\W+(?:\w+\W+){0,5}$", left):
            continue
        if re.search(r"^\W*(?:\w+\W+){0,6}" + NEG_R, right):
            continue
        if need and not re.search(need, left + " " + right):
            continue
        return 1
    return 0


def extract(text):
    return {k: hit(text, p, need) for k, (p, need) in TERMS.items()}

In [ ]:
KW_ALL = pd.DataFrame([extract(t) for t in train.Report])[LABELS].values
print(f'keyword extractor over all {len(train):,} reports: '
      f'{KW_ALL.sum():,} positives, {(KW_ALL.sum(1) == 0).mean():.1%} of studies get nothing')

## 2. Ask the question instead of listing the words

Natural language inference asks: given this text, is this statement true, false, or unsupported?
`mDeBERTa-v3-base-xnli` is trained on 2.7M inference pairs across 100 languages, and it is
*cross-lingual* -- the premise can be a Spanish report while the hypothesis stays in English.

So instead of a Spanish word list for meniscal tears, the question is just
**"The medial meniscus is torn."**, asked against `Rotura de menisco interno.`

Two details do most of the work:

**Score on the entail-vs-contradict axis, dropping neutral.** `softmax(logits[[entail, contra]])[0]`
is the affirm/deny axis. A report saying *no effusion* produces contradiction, not low entailment --
negation is handled by the thing the model was trained to do, in every language at once, with no
negation window to maintain.

**Note the device line printed above.** At the time of writing, Kaggle's PyTorch is built
for `sm_86 / sm_90 / sm_100 / sm_120`, and the GPU it hands out here is a **Tesla P100, which
is `sm_60`**. PyTorch dropped Pascal. The mismatch does not surface at `.to('cuda')` or in
`torch.cuda.is_available()` -- both succeed -- it surfaces as `no kernel image is available`
on the first matmul, which is why the cell above runs one deliberately. If that probe fails,
the scoring pass below reads its scores from
[a published cache](https://www.kaggle.com/datasets/nekkon/rsna-knee-nli-entailment-scores)
of this same code instead of spending thirteen CPU-hours reproducing them. Anything in this
competition that quietly assumes a working GPU for a transformer is worth re-checking.

**Ask sentence by sentence, take the maximum.** A 980-character report containing one relevant
clause dilutes to nothing as a single premise. Section 3 measures this: whole-report premises
score around 0.60 mean AUC, sentence-level maxima around 0.73. The hypotheses below were written from
the finding names before looking at the 58 labelled studies.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_DIR = os.path.dirname(glob.glob('/kaggle/input/**/config.json', recursive=True)[0])
print('model:', MODEL_DIR)

HYP = {
    'ACL':              'The anterior cruciate ligament is torn.',
    'MCL':              'The medial collateral ligament is injured.',
    'Medial Meniscus':  'The medial meniscus is torn.',
    'Lateral Meniscus': 'The lateral meniscus is torn.',
    'Medial OA':        'There is osteoarthritis of the medial compartment of the knee.',
    'Lateral OA':       'There is osteoarthritis of the lateral compartment of the knee.',
    'PF OA':            'There is patellofemoral osteoarthritis.',
    'Effusion':         'There is a joint effusion in the knee.',
    'Synovitis':        'There is synovitis.',
    "Baker's":          "There is a Baker's cyst.",
    'Contusion':        'There is bone marrow oedema or a bone contusion.',
    'Fracture':         'There is a fracture.',
}
assert set(HYP) == set(LABELS)

_SPLIT = re.compile(r'(?<=[.;:!?])\s+|\n+')

def sentences(report, max_n=60):
    # Keep every non-empty fragment. Filtering short ones costs AUC: the one-word Spanish
    # clause "Derrame." IS the effusion finding, and a >=4-word filter throws it away
    # (it costs about 0.03 mean AUC on the 58). Section headers score near zero anyway, so the
    # per-study maximum ignores them without being told to.
    out = [s.strip() for s in _SPLIT.split(str(report))]
    return ([s for s in out if s] or [str(report)[:400]])[:max_n]

tok = AutoTokenizer.from_pretrained(MODEL_DIR)
mdl = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR).eval()


def pick_device():
    '''Kaggle hands out whichever GPU is free, and a torch build without kernels for
    that architecture fails on the first matmul with "no kernel image is available"
    rather than at .to(). So actually run one.'''
    print('torch', torch.__version__, '| built for', torch.cuda.get_arch_list()[-4:]
          if torch.cuda.is_available() else '(no cuda)')
    if not torch.cuda.is_available():
        return 'cpu'
    name = torch.cuda.get_device_name(0)
    cap = torch.cuda.get_device_capability(0)
    print(f'gpu {name}, compute capability sm_{cap[0]}{cap[1]}')
    try:
        (torch.zeros(8, 8, device='cuda') @ torch.zeros(8, 8, device='cuda')).cpu()
        return 'cuda'
    except Exception as e:
        print(f'!! cuda unusable ({type(e).__name__}), falling back to cpu:',
              str(e).splitlines()[0])
        return 'cpu'


dev = pick_device()
mdl.to(dev)
# deliberately fp32 -- DeBERTa-v3's disentangled attention can overflow in half
# precision, and a silent NaN here would read as a finding rather than a bug
i2l = {v.lower(): k for k, v in mdl.config.id2label.items()}
I_ENT, I_CON = i2l['entailment'], i2l['contradiction']
print('entail idx', I_ENT, ' contradict idx', I_CON)

In [ ]:
@torch.no_grad()
def affirm(premises, hypothesis, batch=256):
    '''P(entail) on the entail-vs-contradict axis, one score per premise.'''
    hyp = [hypothesis] * len(premises)
    out = np.empty(len(premises), dtype=np.float32)
    for i in range(0, len(premises), batch):
        enc = tok(premises[i:i + batch], hyp[i:i + batch], truncation=True,
                  padding=True, max_length=96, return_tensors='pt').to(dev)
        lg = mdl(**enc).logits.float()
        out[i:i + batch] = torch.softmax(lg[:, [I_ENT, I_CON]], -1)[:, 0].cpu().numpy()
    return out


demo = [
    ('Rotura de menisco interno.',            'Medial Meniscus'),   # Spanish, affirmed
    ('Menisco interno sin alteraciones.',     'Medial Meniscus'),   # Spanish, denied
    ('Efuzyon izlenmedi.',                    'Effusion'),          # Turkish, negation AFTER
    ('Eklem icinde belirgin efuzyon mevcut.', 'Effusion'),          # Turkish, affirmed
    ('Geen aanwijzingen voor een bakercyste.', "Baker's"),          # Dutch, denied
    ('Kleine bakercyste in de fossa poplitea.', "Baker's"),         # Dutch, affirmed
]
print(f'{"premise":42s} {"finding":17s} {"affirmed":>9s}')
for text, lab in demo:
    print(f'{text[:41]:42s} {lab:17s} {affirm([text], HYP[lab])[0]:9.3f}')

Three languages, three negation grammars, no vocabulary anywhere. The Turkish pair is the
interesting one: the same word `efuzyon` appears in both, and only the trailing `izlenmedi`
separates them.

In [ ]:
# Reports are templated, so the same sentence recurs across studies -- score each unique
# sentence once. 82k fragments collapse to ~34k unique, a 2.4x saving on a 12-hypothesis pass.
per_study = [sentences(r) for r in train.Report]
uniq = sorted({s for ss in per_study for s in ss}, key=len)      # length-sorted: less padding
u_index = {s: i for i, s in enumerate(uniq)}
owner = [np.array([u_index[s] for s in ss]) for ss in per_study]
print(f'{sum(len(s) for s in per_study):,} sentences -> {len(uniq):,} unique')

# ~400k forward passes. That is 30-40 minutes on a working GPU and roughly 13 hours on
# CPU, which is past the session limit -- so if the probe above fell back to CPU, read the
# cached scores instead. The cache is this exact loop's output, published as a dataset so
# the notebook still produces its results on a session that got a GPU torch cannot drive.
CACHE = glob.glob('/kaggle/input/**/nli_entailment_scores.csv', recursive=True)

import time
t0 = time.time()
if dev == 'cpu' and CACHE:
    print('running on cpu -- loading cached entailment scores from', CACHE[0])
    cached = pd.read_csv(CACHE[0]).set_index('StudyInstanceUID').loc[train.StudyInstanceUID]
    NLI = cached[LABELS].to_numpy(dtype=np.float32)
    SCORED_HERE = False
else:
    NLI = np.zeros((len(train), len(LABELS)), dtype=np.float32)
    for j, lab in enumerate(LABELS):
        sc = affirm(uniq, HYP[lab])
        NLI[:, j] = [sc[ix].max() for ix in owner]
        print(f'  {lab:18s} {time.time() - t0:6.1f}s')
    SCORED_HERE = True

np.save('nli_scores.npy', NLI)
assert NLI.shape == (len(train), len(LABELS)) and np.isfinite(NLI).all()
print(f'{"scored" if SCORED_HERE else "loaded"} {NLI.shape} in {time.time() - t0:.0f}s')

## 3. Sentence by sentence, not report by report

The obvious implementation feeds the whole report as the premise. It is also the wrong one, and
the gap is large enough to be worth a cell of its own.

In [ ]:
from sklearn.metrics import roc_auc_score

G = gold[LABELS].values.astype(int)
gi = train.StudyInstanceUID.isin(gold.StudyInstanceUID).values
NLI_G = NLI[gi]                                  # sentence-max scores for the 58

# 58 reports x 12 hypotheses is 696 passes -- cheap enough to run even on cpu
full_prem = [str(r)[:2000] for r in gold.Report]
FULL_G = np.column_stack([affirm(full_prem, HYP[l]) for l in LABELS])

ab = pd.DataFrame({
    'finding': LABELS,
    'whole report': [roc_auc_score(G[:, j], FULL_G[:, j]) for j in range(len(LABELS))],
    'sentence max': [roc_auc_score(G[:, j], NLI_G[:, j]) for j in range(len(LABELS))],
})
print(ab.round(3).to_string(index=False))
print(f"\nmean AUC   whole report {ab['whole report'].mean():.3f}"
      f"   sentence max {ab['sentence max'].mean():.3f}")

A knee MRI report is a list of independent observations about eight or nine structures. Asked as
one premise, the model has to decide whether a 980-character list entails a statement about one
of them, and the answer is always "sort of". Asked one clause at a time, the question is the one
the model was trained on.

Everything below uses the sentence-level maxima.

## 4. Where entailment wins, and where it does not

The keyword extractor is binary, so its ROC has a single interior point and its AUC equals its
balanced accuracy exactly. That makes AUC a like-for-like comparison between a binary rule and a
continuous score, with no threshold to choose on either side.

58 studies is a small test set, so every difference below carries a bootstrap interval. The
verdict column only says NLI or keyword when the 95% interval excludes zero.

In [ ]:
KW_G = pd.DataFrame([extract(t) for t in gold.Report])[LABELS].values
rng = np.random.default_rng(0)
B, n = 4000, len(gold)

rows = []
for j, lab in enumerate(LABELS):
    g, s, k = G[:, j], NLI_G[:, j], KW_G[:, j]
    a_nli, a_kw = roc_auc_score(g, s), roc_auc_score(g, k)
    d = []
    for _ in range(B):
        i = rng.integers(0, n, n)
        if 0 < G[i, j].sum() < n:
            d.append(roc_auc_score(G[i, j], s[i]) - roc_auc_score(G[i, j], k[i]))
    d = np.array(d)
    lo, hi = np.quantile(d, [.025, .975])
    rows.append({'finding': lab, 'pos': int(g.sum()), 'keyword': a_kw, 'NLI': a_nli,
                 'delta': a_nli - a_kw, '95% lo': lo, '95% hi': hi,
                 'verdict': 'NLI' if lo > 0 else ('keyword' if hi < 0 else 'tie')})
res = pd.DataFrame(rows).sort_values('delta', ascending=False)
print(res.round(3).to_string(index=False))
print(f"\nmean AUC   keyword {res.keyword.mean():.3f}   NLI {res.NLI.mean():.3f}")
print('verdicts:', res.verdict.value_counts().to_dict())

In [ ]:
fig, ax = plt.subplots(figsize=(9.5, 5.6))
r = res.sort_values('NLI')
y = np.arange(len(r))
for yi, (_, row) in zip(y, r.iterrows()):
    c = {'NLI': NLIC, 'keyword': KWC, 'tie': '#b6b9be'}[row.verdict]
    ax.plot([row.keyword, row.NLI], [yi, yi], color=c, lw=2.4, zorder=1,
            solid_capstyle='round')
ax.scatter(r.keyword, y, s=52, color=KWC, zorder=3, label='keyword (v1)')
ax.scatter(r.NLI, y, s=52, color=NLIC, zorder=3, label='entailment (v2)')
ax.axvline(.5, color=GRID, lw=1, zorder=0)
ax.set_yticks(y); ax.set_yticklabels(r.finding)
for yi, (_, row) in zip(y, r.iterrows()):
    if row.verdict != 'tie':
        ax.text(max(row.keyword, row.NLI) + .012, yi, row.verdict, va='center', fontsize=9,
                color=NLIC if row.verdict == 'NLI' else KWC, fontweight='bold')
ax.set_xlim(.40, 1.0); ax.set_xlabel('AUC on the 58 labelled studies')
ax.set_title('Entailment moves the meniscus findings and loses the compartment ones')
ax.legend(frameon=False, loc='lower right')
ax.text(.40, -1.35, 'grey = the 95% bootstrap interval on the difference includes zero',
        fontsize=9, color=MUTED)
plt.tight_layout(); plt.show()

Three findings improve beyond the noise, one gets worse beyond the noise, and eight are
indistinguishable on 58 studies.

**Medial meniscus, +0.32.** The largest single movement in either notebook, and the correction
to v1's claim that this finding was near-unrecoverable. It was recoverable; the regex could not
reach it.

**Lateral OA, -0.19.** Entailment lands *below chance* here. That is not the model being
mediocre, it is the model being unable to answer the question at all, and section 5 shows why.

**Eight ties.** Not "no difference" -- "58 studies cannot tell". Effusion at +0.11 with an
interval of [-0.06, +0.27] may well be a real improvement that this test set cannot resolve.

## 5. Which method actually knows the side

Four of the twelve findings are the same pathology on two sides: medial and lateral meniscus,
medial and lateral compartment OA. A model that fires on "there is a meniscal tear" without
resolving *which* meniscus will still post a good AUC on both, because the two labels co-occur
in 12 of the 58 studies.

The test for that is the cross-AUC: score with the **medial** hypothesis, then measure how well
it predicts the **lateral** label. A method that binds the side should be at chance off-diagonal.

In [ ]:
PAIRS = [('Medial Meniscus', 'Lateral Meniscus'), ('Medial OA', 'Lateral OA')]
ix = {l: i for i, l in enumerate(LABELS)}

rows = []
for a, b in PAIRS:
    for src, tgt in ((a, b), (b, a)):
        rows.append({'score from': src,
                     'AUC on own label': roc_auc_score(G[:, ix[src]], NLI_G[:, ix[src]]),
                     'AUC on other side': roc_auc_score(G[:, ix[tgt]], NLI_G[:, ix[src]]),
                     'kw own': roc_auc_score(G[:, ix[src]], KW_G[:, ix[src]]),
                     'kw other side': roc_auc_score(G[:, ix[tgt]], KW_G[:, ix[src]])})
lat = pd.DataFrame(rows)
print(lat.round(3).to_string(index=False))

for a, b in PAIRS:
    ya, yb = G[:, ix[a]].astype(bool), G[:, ix[b]].astype(bool)
    print(f'\n{a} / {b} co-occurrence in the 58: '
          f'both {int((ya & yb).sum())}, only {a.split()[0].lower()} {int((ya & ~yb).sum())}, '
          f'only {b.split()[0].lower()} {int((~ya & yb).sum())}, neither {int((~ya & ~yb).sum())}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.3), sharey=True)
for ax, (meth, own_c, oth_c, tag) in zip(
        axes, [('NLI', NLIC, '#f3bda2', 'entailment'), ('kw', KWC, '#a9c8ee', 'keyword')]):
    names = list(lat['score from'])
    own = lat['AUC on own label'] if meth == 'NLI' else lat['kw own']
    oth = lat['AUC on other side'] if meth == 'NLI' else lat['kw other side']
    x = np.arange(len(names))
    ax.bar(x - .19, own, .36, color=own_c, label='predicts its own label')
    ax.bar(x + .19, oth, .36, color=oth_c, label='predicts the other side')
    ax.axhline(.5, color=WARN, lw=1.2, ls=(0, (4, 3)))
    for xi, (a, b) in enumerate(zip(own, oth)):
        ax.text(xi - .19, a + .012, f'{a:.2f}', ha='center', fontsize=9, color=INK)
        ax.text(xi + .19, b + .012, f'{b:.2f}', ha='center', fontsize=9, color=MUTED)
    ax.set_xticks(x)
    ax.set_xticklabels([n.replace(' Meniscus', '\nmeniscus').replace(' OA', '\ncompartment OA')
                        for n in names], fontsize=9)
    ax.set_ylim(0, 1.0); ax.set_title(tag)
    ax.set_yticks([0, .5, 1.0])
axes[0].set_ylabel('AUC'); axes[0].legend(frameon=False, fontsize=9, loc='upper right')
axes[0].text(-.55, .52, 'chance', fontsize=8.5, color=WARN)
fig.suptitle('Does the score know which side it is talking about?',
             x=.078, ha='left', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout(); plt.show()

Read the meniscus rows first. The entailment score for the medial meniscus reaches **0.87 on its
own label and 0.48 on the lateral one** -- exactly chance. It is not detecting "a meniscal tear",
it is answering the question that was asked, in Spanish, Turkish and Greek.

Now read the keyword rows for the same finding: **0.558 on its own label and 0.655 on the other
side.** The v1 extractor predicts the wrong meniscus better than the right one. Its pattern allows
up to two words between the side and the structure in either direction, and in a sentence naming
both menisci that window straddles them. That is a bug in v1, and it is the mechanism behind the
0.558 I published.

The OA rows are the opposite story and the reason for the -0.19. Both entailment scores sit near
chance on both labels: given a clause about arthrosis of a compartment, the model cannot attach
the modifier to the finding. The keyword extractor's explicit side-word requirement is crude, but
crude is enough here, and it wins.

**Neither method is better. They fail in complementary ways** -- entailment reads the language and
cannot always bind a modifier; regex binds the modifier by construction and cannot read the
language.

## 6. The threshold wall

Ranking better is not the same as labelling better. A blend of the two scores ranks best of all,
so the natural next step is to threshold it per finding, fitted on the 58 labelled studies, and
ship binary labels like v1 did.

That step gives the entire gain back. Below, each threshold is chosen on 57 studies and applied
to the one held out, so the number is what you would actually get -- not the in-sample optimum.

In [ ]:
from scipy.stats import rankdata

def bal(g, p):
    return (p[g == 1].mean() + 1 - p[g == 0].mean()) / 2

def loo_bal(g, s):
    '''threshold fitted on n-1 studies, scored on the held-out one'''
    grid = np.unique(np.quantile(s.astype(float), np.linspace(.05, .95, 40)))
    pred = np.empty(len(g), int)
    for i in range(len(g)):
        m = np.ones(len(g), bool); m[i] = False
        b = [bal(g[m], (s[m] >= th).astype(int)) for th in grid]
        pred[i] = int(s[i] >= grid[int(np.argmax(b))])
    return bal(g, pred)

BLEND_G = np.column_stack([(rankdata(NLI_G[:, j]) / n + KW_G[:, j]) / 2
                           for j in range(len(LABELS))])

w = pd.DataFrame({
    'finding': LABELS,
    'AUC keyword': [roc_auc_score(G[:, j], KW_G[:, j]) for j in range(len(LABELS))],
    'AUC NLI': [roc_auc_score(G[:, j], NLI_G[:, j]) for j in range(len(LABELS))],
    'AUC blend': [roc_auc_score(G[:, j], BLEND_G[:, j]) for j in range(len(LABELS))],
    'binary keyword': [bal(G[:, j], KW_G[:, j]) for j in range(len(LABELS))],
    'binary blend, held out': [loo_bal(G[:, j], BLEND_G[:, j]) for j in range(len(LABELS))],
})
print(w.round(3).to_string(index=False))
print('\nmean   ' + '   '.join(f'{c} {w[c].mean():.3f}' for c in w.columns[1:]))

In [ ]:
fig, ax = plt.subplots(figsize=(8.2, 4.6))
groups = ['keyword\n(v1)', 'entailment', 'blend']
auc = [w['AUC keyword'].mean(), w['AUC NLI'].mean(), w['AUC blend'].mean()]
bina = [w['binary keyword'].mean(), np.nan, w['binary blend, held out'].mean()]
x = np.arange(3)
ax.bar(x - .19, auc, .36, color=BLEND, label='as a ranking (AUC)')
ax.bar(x + .19, bina, .36, color='#c2c5ca', label='as binary labels, threshold held out')
for xi, v in zip(x, auc):
    ax.text(xi - .19, v + .006, f'{v:.3f}', ha='center', fontsize=10, color=INK)
for xi, v in zip(x, bina):
    if not np.isnan(v):
        ax.text(xi + .19, v + .006, f'{v:.3f}', ha='center', fontsize=10, color=MUTED)
ax.annotate('', xy=(2 - .19, auc[2] - .004), xytext=(0 - .19, auc[0] + .004),
            arrowprops=dict(arrowstyle='->', color=BLEND, lw=1.6,
                            connectionstyle='arc3,rad=-.28'))
ax.text(1 - .30, .795, f'+{auc[2] - auc[0]:.3f} AUC', color=BLEND, fontsize=10,
        fontweight='bold', ha='center')
ax.text(2 + .19, bina[2] - .055, f'{bina[2] - bina[0]:+.3f}\nas labels', ha='center',
        fontsize=10, color=WARN, fontweight='bold')
ax.set_xticks(x); ax.set_xticklabels(groups)
ax.set_ylim(.5, .82); ax.set_ylabel('mean over the 12 findings')
ax.set_title('The ranking improves; the labels do not')
ax.legend(frameon=False, fontsize=9.5, loc='upper left')
plt.tight_layout(); plt.show()

Read the last two columns against each other. Roughly eight points of ranking gain arrive as
about one point of labelling gain -- the binarised blend lands within noise of the v1 keyword
baseline it was meant to beat.

Twelve thresholds fitted on 58 studies is roughly five positive examples per threshold. The
threshold that is optimal on 57 studies is not optimal on the 58th, and the variance of that
choice is larger than the gain you were trying to capture. This is not fixable by a better
search -- it is the size of the labelled set.

Three consequences for anyone training on this output:

1. **Take the probabilities, not a binarisation of them.** Soft targets keep the ranking
   information that a threshold throws away.
2. **Do not spend the 58 studies on threshold fitting.** They are the only clean signal in the
   competition and this is the least valuable thing to do with them.
3. **The competition metric decides the threshold, not the labels.** If it is AUC-like, you never
   need one. If it needs a hard call, tune it on your own model's validation output, downstream
   of everything, where you have more than 58 rows.

## 6b. The contrastive fix, tested

The closing section of the previous version promised an untested fix for the compartment-OA
failure: ask a *pair* of side hypotheses and use the difference, instead of scoring each side
independently. Here it is, tested. For each sentence, score both sides and combine as
`s_own * (s_own - s_other + 1) / 2` -- the entailment strength, discounted by how much the
sentence also affirms the opposite side -- then take the per-report maximum as before.

It moves exactly the two findings it was aimed at, and nothing it was not:

- **Medial OA 0.602 -> 0.785, Lateral OA 0.471 -> 0.709.** Both now clear the keyword floor
  (0.720 / 0.661) that section 4 reported them losing to.
- **The menisci get *worse* under the same treatment** (medial 0.871 -> 0.743), so the fix is
  applied only to the OA pair. The meniscus hypotheses already bind the side; subtracting the
  other side just adds noise to a solved problem.
- **Honesty about the intervals:** the paired bootstrap gives [-0.03, +0.40] and [-0.00, +0.48]
  -- large deltas, formally not significant. That is the companion notebook's power curve doing exactly
  what it said: a +0.2 improvement on 11-15 positives sits at the edge of what 58 studies can see.
- One asymmetry survives: the medial-OA contrast score still predicts the lateral label almost
  as well as its own, so laterality binding for OA is only half-fixed. The magnitude question
  ("is there compartment OA?") is answered; the side question is partly open.

`weak_labels_v2.csv` now carries four extra `_contrast` columns for these findings.

In [ ]:
# Contrastive scores: computed live on a working GPU, else loaded from the same cache dataset.
CPATH = glob.glob('/kaggle/input/**/nli_contrast_scores.csv', recursive=True)
if dev != 'cpu':
    def contrast_pair(a, b):
        sa, sb = affirm(uniq, HYP[a]), affirm(uniq, HYP[b])
        soft_a, soft_b = sa * (sa - sb + 1) / 2, sb * (sb - sa + 1) / 2
        return ([max(soft_a[u_index[s]] for s in ss) for ss in per_study],
                [max(soft_b[u_index[s]] for s in ss) for ss in per_study])
    CON = {}
    for a, b in [('Medial Meniscus', 'Lateral Meniscus'), ('Medial OA', 'Lateral OA')]:
        CON[a], CON[b] = contrast_pair(a, b)
    CON = pd.DataFrame(CON, index=train.index)
else:
    CON = pd.read_csv(CPATH[0]).set_index('StudyInstanceUID')
    CON = CON.loc[train.StudyInstanceUID].reset_index(drop=True)
    CON.columns = [c.replace('_contrast', '') for c in CON.columns]

con_g = CON[gi].reset_index(drop=True)
rows = []
for lab in ['Medial Meniscus', 'Lateral Meniscus', 'Medial OA', 'Lateral OA']:
    j = LABELS.index(lab)
    rows.append({'finding': lab,
                 'keyword': roc_auc_score(G[:, j], KW_G[:, j]),
                 'plain NLI': roc_auc_score(G[:, j], NLI_G[:, j]),
                 'contrast': roc_auc_score(G[:, j], con_g[lab].values)})
print(pd.DataFrame(rows).round(3).to_string(index=False))
print('\ncontrast helps the OA pair, hurts the menisci -- applied to OA only in the output')

## 7. The output

`weak_labels_v2.csv`, one row per study, and for each of the twelve findings three columns:

| column | what it is |
|---|---|
| `<finding>` | the blend -- rank-averaged NLI and keyword, in [0, 1]. Use this one. |
| `<finding>_nli` | raw entailment score, the affirm-vs-deny probability |
| `<finding>_kw` | the v1 keyword binary, kept so you can reproduce or override |

Plus `has_gold` (whether this study is one of the 58) and, in the printed table below,
a per-finding `AUC` you can use as a loss weight.

In [ ]:
BLEND = np.column_stack([(rankdata(NLI[:, j]) / len(train) + KW_ALL[:, j]) / 2
                         for j in range(len(LABELS))])

out = pd.DataFrame({'StudyInstanceUID': train.StudyInstanceUID})
for j, lab in enumerate(LABELS):
    out[lab] = BLEND[:, j].round(4)
    out[f'{lab}_nli'] = NLI[:, j].round(4)
    out[f'{lab}_kw'] = KW_ALL[:, j].astype(int)
for lab in CON.columns:
    out[f'{lab}_contrast'] = np.asarray(CON[lab]).round(4)
out['has_gold'] = train.StudyInstanceUID.isin(gold.StudyInstanceUID).values
out.to_csv('weak_labels_v2.csv', index=False)

quality = pd.DataFrame({
    'finding': LABELS,
    'AUC of the blend': [roc_auc_score(G[:, j], BLEND_G[:, j]) for j in range(len(LABELS))],
    'use as loss weight': [round(max(0., 2 * roc_auc_score(G[:, j], BLEND_G[:, j]) - 1), 3)
                           for j in range(len(LABELS))],
}).sort_values('AUC of the blend', ascending=False)
print(quality.round(3).to_string(index=False))
print(f'\nweak_labels_v2.csv  {out.shape[0]:,} rows x {out.shape[1]} columns')
out.head(3)

## What changed, and what I got wrong

**v1 said the meniscus findings were near-unrecoverable from text.** They are not. Medial meniscus
goes 0.558 -> 0.87 AUC, and the cross-AUC check shows the gain is real laterality binding, not
co-occurrence. The 0.558 was a bug in my own regex, which predicts the wrong meniscus better than
the right one.

**v1 shipped binary labels.** That was the wrong output format. Section 6 shows the binarisation
step destroys the improvement it is applied to; the probabilities are strictly more useful and
cost nothing to ship.

**What still stands from v1.** The 58 labelled studies are not a random sample -- ACL is positive
in 41% of them against a corpus rate near 20% -- so they remain unusable for setting an output
prior. And the keyword floor is a floor: it still wins on compartment OA, decisively.

**The promised fix is now tested** (section 6b): a contrastive pair of hypotheses lifts
compartment OA above the keyword floor -- the one place entailment was losing -- while the same
treatment hurts the menisci and is therefore not applied to them. With it, every finding's best
text score clears 0.66. Side-binding for OA remains only half-fixed, and the deltas sit at the
edge of what 58 studies can resolve; both caveats are in the section.

Measurements here are on 58 studies. Treat every interval in section 4 as the real precision of
every claim above, including mine.